In [2]:
import asyncio
from pylabrobot.liquid_handling.backends.tecan.EVO_backend import EVOBackend, LiHa, RoMa, PnP
from pylabrobot.liquid_handling.backends.tecan.EVO_backend import EVOArm
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.resources.tecan import EVO200Deck
from pylabrobot.liquid_handling.backends.tecan import arm_commands

In [ ]:
# Detect Arms(Don't Move)
async def detect_arms():
    backend = EVOBackend()
    await backend.io.setup()
    pnp  = await backend.setup_arm(EVOBackend.PNP)
    liha = await backend.setup_arm(EVOBackend.LIHA)
    #roma = await backend.setup_arm(EVOBackend.ROMA)

    print(f"PnP connected:  {pnp}")
    print(f"LiHa connected: {liha}")
    #print(f"RoMa connected: {roma}")

# Report Z-Axis Range on All Arms
async def report_z_ranges():
    backend = EVOBackend()

    if await backend.setup_arm(EVOBackend.PNP):
        pnp = PnP(backend, EVOBackend.PNP)
        z_pnp = await pnp.report_z_param(5)
        print(f"PnP Z-axis range: {z_pnp}")

    if await backend.setup_arm(EVOBackend.LIHA):
        liha = LiHa(backend, EVOBackend.LIHA)
        z_liha = await liha.report_z_param(5)
        print(f"LiHa Z-axis range: {z_liha}")

    if await backend.setup_arm(EVOBackend.ROMA):
        roma = RoMa(backend, EVOBackend.ROMA)
        z_roma = await roma.report_z_param(5)
        print(f"RoMa Z-axis range: {z_roma}")

# Moves all arms to Maximum Z Position
async def move_arms_max_z():
    backend = EVOBackend()

    """
    Move all connected arms (LiHa, RoMa, PnP) to their maximum Z-axis height.
    """
    # Move LiHa to max Z
    if backend.liha_connected:
        z_range_liha = await backend.liha.report_z_param(5)  # 5 = actual machine range
        if isinstance(z_range_liha, list):
            z_range_liha = z_range_liha[0]
        await backend.liha.set_z_travel_height([z_range_liha] * backend.num_channels)
        await backend.liha.position_absolute_all_axis(
            x=None, y=None, ys=None, z=[z_range_liha] * backend.num_channels    #z is a list and must be passed a height for each pipetting head
        )

    # Move RoMa to max Z
    if backend.roma_connected:
        z_range_roma = await backend.roma.report_z_param(5)
        # RoMa uses vector coordinate movement, so just move Z
        await backend.roma.set_vector_coordinate_position(
            v=1, x=None, y=None, z=z_range_roma, r=None, g=None, speed=1
        )
        await backend.roma.action_move_vector_coordinate_position()

    # Move PnP to max Z
    if backend.pnp_connected:
        z_range_pnp = await backend.pnp.report_z_param(5)
        await backend.pnp.position_absolute_all(z=z_range_pnp)

# Moves Arms to Starting Positions
async def move_arms_start():
    lh = LiquidHandler(backend = EVOBackend(), deck=EVO200Deck())
    await lh.setup()


    # Always move to max Z first for safety
    #await move_arms_max_z()

    # Now move each arm to its defined starting position (XY and other axes)
    if lh.backend.liha_connected:
        z_range_liha = await lh.backend.liha.report_z_param(5)
        if isinstance(z_range_liha, list):
            z_range_liha = z_range_liha[0]
        # Move LiHa to (x=9241, y=793) and keep Z at max for safety
        await lh.backend.liha.set_z_travel_height([z_range_liha] * lh.backend.num_channels)
        await lh.backend.liha.position_absolute_all_axis(
            x=9241,
            y=793,
            ys=None,
            z=[z_range_liha] * lh.backend.num_channels
        )

    if backend.roma_connected:
        z_range_roma = await backend.roma.report_z_param(5)
        # Move RoMa to (x=14628, y=1999, r=1800, g=900), Z at max
        await backend.roma.set_vector_coordinate_position(
            v=1,
            x=14628,
            y=1999,
            z=z_range_roma,
            r=1800,
            g=900,
            speed=1
        )
        await backend.roma.action_move_vector_coordinate_position()

    if lh.backend.pnp_connected:
        z_range_pnp = await lh.backend.pnp.report_z_param(5)
        # Move PnP to (x=-100, y=-700, r=0, g=280), Z at max
        await lh.backend.pnp.position_absolute_all(
            x=-100,
            y=-700,
            z=z_range_pnp,
            r=0,
            g=280
        )




In [ ]:
#await detect_arms()
#await asyncio.sleep(1)
#await report_z_ranges()
#await asyncio.sleep(1)
#await move_arms_max_z()
#await asyncio.sleep(1)
await move_arms_start()
#await asyncio.sleep(1)

In [ ]:
#Change Implemented in EVO_backend.py
class PnP(EVOArm):
    """
    Provides firmware commands for the Pick & Place (PnP) arm.
    """

    async def fake_initialize(self):
        """Fake initialization of all axes (no physical movement)."""
        await self.backend.send_command(module=self.module, command="PIF")

    async def initialize(self):
        """Initialize all axes (X/Y/Z/R/G). Required before any movement."""
        await self.backend.send_command(module=self.module, command="PIA")

    async def initialize_x(self, speed: int = None):
        """Initialize X-axis, optional override for speed (0.1 mm/s)."""
        await self.backend.send_command(self.module, "PIX", [speed])

    async def initialize_y(self, speed: int = None):
        """Initialize Y-axis, optional override for speed (0.1 mm/s)."""
        await self.backend.send_command(self.module, "PIY", [speed])

    async def initialize_z(self, speed: int = None):
        """Initialize Z-axis, optional override for speed (0.1 mm/s)."""
        await self.backend.send_command(self.module, "PIZ", [speed])

    async def initialize_dependent_axes(self):
        """Initialize Z, Y, Rotator, and Gripper axes."""
        await self.backend.send_command(self.module, "PIR")

    # Absolute positioning for individual axes
    async def position_absolute_x(self, x: int):
        await self.backend.send_command(self.module, "PAX", [x])

    async def position_absolute_y(self, y: int):
        await self.backend.send_command(self.module, "PAY", [y])

    async def position_absolute_z(self, z: int):
        await self.backend.send_command(self.module, "PAZ", [z])

    async def position_absolute_r(self, r: int):
        await self.backend.send_command(self.module, "PAR", [r])

    async def position_absolute_g(self, g: int):
        await self.backend.send_command(self.module, "PAG", [g])

    # Relative positioning for individual axes
    async def position_relative_x(self, x: int):
        await self.backend.send_command(self.module, "PRX", [x])

    async def position_relative_y(self, y: int):
        await self.backend.send_command(self.module, "PRY", [y])

    async def position_relative_z(self, z: int):
        await self.backend.send_command(self.module, "PRZ", [z])

    async def position_relative_r(self, r: int):
        await self.backend.send_command(self.module, "PRR", [r])

    async def position_relative_g(self, g: int):
        await self.backend.send_command(self.module, "PRG", [g])

    # Simultaneous axis movement
    async def position_absolute_all(self, x=None, y=None, z=None, r=None, g=None):
        params = [x, y, z, r, g]
        await self.backend.send_command(self.module, "PAA", params)

    async def position_relative_all(self, x=None, y=None, z=None, r=None, g=None):
        params = [x, y, z, r, g]
        await self.backend.send_command(self.module, "PRA", params)

    async def move_absolute_slow(self, x=None, y=None, z=None, r=None, g=None):
        params = [x, y, z, r, g]
        await self.backend.send_command(self.module, "MAA", params)

    async def move_relative_slow(self, x=None, y=None, z=None, r=None, g=None):
        params = [x, y, z, r, g]
        await self.backend.send_command(self.module, "MRA", params)

    # Gripper and tube handling
    async def grip_tube(self, grip_search: int, grip_open: int):
        await self.backend.send_command(self.module, "AGR", [grip_search, grip_open])

    async def pick_tube(self, grip_search: int, grip_open: int, z_pick: int, z_up: int, rotation: int, rot_comp: int):
        params = [grip_search, grip_open, z_pick, z_up, rotation, rot_comp]
        await self.backend.send_command(self.module, "APC", params)

    async def place_tube(self, grip_open: int, z_place: int, z_up: int, rotation: int, tube_check: int, wipe_off_dist: int, tube_left_check: int):
        params = [grip_open, z_place, z_up, rotation, tube_check, wipe_off_dist, tube_left_check]
        await self.backend.send_command(self.module, "APL", params)

    # Teach diameter
    async def teach_diameter(self, diameter: int):
        await self.backend.send_command(self.module, "ATD", [diameter])

    # Auto-range commands
    async def auto_range_x(self):
        await self.backend.send_command(self.module, "ARX")

    async def auto_range_y(self):
        await self.backend.send_command(self.module, "ARY")

    async def auto_range_z(self):
        await self.backend.send_command(self.module, "ARZ")

    # Set commands (examples, extend as needed)
    async def set_grip_params(self, pwm_limit: int, grip_speed: int, min_space: int, max_space: int):
        await self.backend.send_command(self.module, "SGP", [pwm_limit, grip_speed, min_space, max_space])

    async def set_fast_speed_x(self, end_speed: int, accel: int):
        await self.backend.send_command(self.module, "SFX", [end_speed, accel])

    async def set_fast_speed_y(self, end_speed: int, accel: int):
        await self.backend.send_command(self.module, "SFY", [end_speed, accel])

    async def set_fast_speed_z(self, end_speed: int, accel: int):
        await self.backend.send_command(self.module, "SFZ", [end_speed, accel])

    async def set_fast_speed_r(self, end_speed: int, accel: int):
        await self.backend.send_command(self.module, "SFR", [end_speed, accel])

    # Report commands
    async def report_x_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RPX", [selector])
        return resp["data"][0]

    async def report_y_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RPY", [selector])
        return resp["data"][0]

    async def report_z_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RPZ", [selector])
        return resp["data"][0]

    async def report_r_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RPR", [selector])
        return resp["data"][0]

    async def report_g_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RPG", [selector])
        return resp["data"][0]

    async def report_gripper_param(self, selector: int) -> int:
        resp = await self.backend.send_command(self.module, "RGP", [selector])
        return resp["data"][0]

    async def report_tube_diameter(self) -> int:
        resp = await self.backend.send_command(self.module, "RTD")
        return resp["data"][0]

    async def report_displacement_x(self) -> int:
        resp = await self.backend.send_command(self.module, "RXD")
        return resp["data"][0]
